In [1]:
import sys
import pandas as pd
from pathlib import Path

# This moves from notebooks/ up one level to project root
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Using project root:", PROJECT_ROOT)


Using project root: C:\Users\nathan.taylor\Jupyter Notebooks\outlier_pipeline


In [2]:
from src.runner import run_all_jobs

CONFIG_PATH = PROJECT_ROOT / "configs" / "jobs_021926.yml"

results = run_all_jobs(str(CONFIG_PATH))


2026-02-19 08:48:36,454 [INFO] src.runner - Starting job: ATT_MOB_hold_time
2026-02-19 08:48:36,685 [INFO] pyhive.presto - SELECT 
--CAST(week_stop_date AS DATE) AS week_,
--date,
expert_id,
-- year_month,
metric,
icp_client,
--tenure_group,
site,
SUM(numerator) AS num,
SUM(denominator) AS den,
ROUND(
COALESCE(
CAST(SUM(numerator) AS DOUBLE) /
NULLIF(CAST(SUM(denominator) AS DOUBLE), 0.0),
0.000
),
3
) AS calc
FROM 
hive.care.expert_performance_metrics a
LEFT OUTER JOIN 
hive.care.l4_asurion_umt_ppx_pay_calendar d ON a."date" = CAST(d.event_date AS DATE)
WHERE 
LOWER(metric) = 'hold time'
AND icp_client = 'MOB-AT&T'
AND a."date" between DATE '2026-02-06' and DATE '2026-02-20' 
 --and expert_id in ()
GROUP BY 1, 2, 3, 4


2026-02-19 08:48:42,939 [INFO] src.runner - [ATT_MOB_hold_time] Raw query rows: 593
2026-02-19 08:48:42,971 [INFO] src.runner - [ATT_MOB_hold_time] After validity filters rows: 577
2026-02-19 08:48:42,989 [INFO] src.runner - [ATT_MOB_hold_time] After eligibility filter

In [3]:
print("\n===== JOB SUMMARY =====\n")

summary_df = pd.DataFrame([
    {
        "job_name": job_name,
        "rows_raw": meta.get("rows_raw"),
        "rows_valid": meta.get("rows_valid"),
        "rows_eligible": meta.get("rows_eligible"),
        "rows_out": meta.get("rows_out"),
        "status": meta.get("status"),
        "output_file": meta.get("out_path"),
    }
    for job_name, meta in results.items()
])

summary_df



===== JOB SUMMARY =====



,job_name,rows_raw,rows_valid,rows_eligible,rows_out,status,output_file
0,ATT_MOB_hold_time,593,577,520,131,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
1,ATT_MOB_talk_time,593,577,520,131,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
2,ATT_MOB_smart_offer,672,541,541,124,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
3,ATT_MOB_OCR,552,529,476,119,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
4,ATT_MOB_Transfers,593,516,465,52,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
5,Mcafee_CRT,165,157,157,125,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
6,Mcafee_Cancel,154,150,150,32,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
7,VZW_MOB_hold_time,476,464,415,104,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
8,VZW_MOB_talk_time,476,464,415,104,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
9,VZW_MOB_smart_offer,520,471,471,71,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
